In [1]:
"""데모 메인 — Jupyter 노트북처럼 셀 단위로 실행 가능.

VSCode / Jupyter에서 `# %%` 블록이 셀로 인식된다.
Ctrl/Cmd+Enter 로 셀을 하나씩 실행하며 시연한다.

Prerequisites:
    pip install playwright requests
    playwright install chromium

    # 모델 서빙 (별도 터미널)
    # python src/inference.py --model nemotron-4b --checkpoint output/sft-nemotron-4b/final --serve
    # 또는 vLLM 직접:
    # vllm serve <checkpoint> --served-model-name nemotron-match --port 8000
"""

'데모 메인 — Jupyter 노트북처럼 셀 단위로 실행 가능.\n\nVSCode / Jupyter에서 `# %%` 블록이 셀로 인식된다.\nCtrl/Cmd+Enter 로 셀을 하나씩 실행하며 시연한다.\n\nPrerequisites:\n    pip install playwright requests\n    playwright install chromium\n\n    # 모델 서빙 (별도 터미널)\n    # python src/inference.py --model nemotron-4b --checkpoint output/sft-nemotron-4b/final --serve\n    # 또는 vLLM 직접:\n    # vllm serve <checkpoint> --served-model-name nemotron-match --port 8000\n'

# Product Matching Demo

1. 사전에 정한 SKU 쿼리 선택
2. 해당 커머스 사이트를 브라우저로 열고 검색
3. 결과 카드 파싱 → 학습된 로컬 모델로 매칭 판정
4. 매칭된 후보를 하이라이트

In [2]:
import json
from dataclasses import asdict
from pathlib import Path

from IPython.display import HTML, display
from playwright.sync_api import sync_playwright

from scraper import ADAPTERS
from model_client import MatchModel, MockMatchModel

# 경로
ROOT = Path(__file__).parent if "__file__" in globals() else Path.cwd()
CACHE_DIR = ROOT / "cache"
QUERIES = json.loads((ROOT / "queries.json").read_text(encoding="utf-8"))["queries"]

# 모델 엔드포인트 — 서빙이 없으면 MockMatchModel 사용 (휴리스틱)
USE_MOCK_MODEL = False  # True 로 바꾸면 서빙 없이 드라이런
model = MockMatchModel() if USE_MOCK_MODEL else MatchModel()

print(f"Loaded {len(QUERIES)} queries · model={'mock' if USE_MOCK_MODEL else model.model_name}")

Loaded 5 queries · model=nemotron-match


## Step 1 — 쿼리 선택

`queries.json` 에서 하나를 고른다. 시연 중에 인덱스만 바꾸면 됨.

In [3]:
QUERY_IDX = 0
q = QUERIES[QUERY_IDX]

print(f"▶ ID         : {q['id']}")
print(f"▶ Platform   : {q['platform']}")
print(f"▶ Query      : {q['query']}")
print(f"▶ SKU name   : {q['sku_name']}")
if q.get("notes"):
    print(f"▶ Notes      : {q['notes']}")

▶ ID         : cp-pillow-01
▶ Platform   : coupang
▶ Query      : 데시뉴 항균 누빔 베개커버
▶ SKU name   : 데시뉴 에센스 항균 누빔 베개커버 40x60 50x70 11컬러 2개세트
▶ Notes      : 정답: 동일 브랜드+상품군, 크기·수량 일치 확인 필요


## Step 2 — 브라우저 열고 검색

헤드풀 모드라 창이 뜬다. CAPTCHA/차단 페이지가 나오면 콘솔에
`🔒 CAPTCHA 감지` 가 뜨며 Enter 대기 — 창에서 직접 풀고 Enter.

In [4]:
USE_CACHE_FIRST = False   # True: 캐시 있으면 라이브 스킵 (안전 fallback)
WRITE_CACHE = True        # 성공한 결과는 cache/ 에 저장해둠

playwright = sync_playwright().start()
browser = playwright.chromium.launch(
    headless=False,
    args=["--disable-blink-features=AutomationControlled"],
)
context = browser.new_context(
    viewport={"width": 1280, "height": 900},
    user_agent=(
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/131.0.0.0 Safari/537.36"
    ),
    locale="ko-KR",
)
page = context.new_page()

adapter = ADAPTERS[q["platform"]](cache_dir=CACHE_DIR)
candidates = adapter.search(
    page, q["query"],
    use_cache=USE_CACHE_FIRST,
    write_cache=WRITE_CACHE,
    max_results=12,
)

print(f"✓ {adapter.display_name}: {len(candidates)} candidates")

Error: It looks like you are using Playwright Sync API inside the asyncio loop.
Please use the Async API instead.

## Step 3 — 후보 카드 리치 디스플레이

In [ ]:
def render_candidates(cands, verdicts=None):
    """후보 리스트를 HTML 그리드로. verdicts가 주어지면 matched 는 녹색 테두리."""
    html = ["""
    <style>
      .demo-grid { display:grid; grid-template-columns:repeat(3, 1fr); gap:12px; font-family:Pretendard,sans-serif; }
      .demo-card { border:1px solid #e2e8f0; border-radius:12px; padding:12px; background:#fff; position:relative; }
      .demo-card.matched { border:2px solid #059669; background:#ecfdf5; }
      .demo-card.nomatch { opacity:0.55; }
      .demo-card img { width:100%; height:140px; object-fit:cover; border-radius:8px; background:#f8fafc; }
      .demo-card .title { font-size:13px; line-height:1.4; margin-top:8px; color:#0f172a; font-weight:500; max-height:54px; overflow:hidden; }
      .demo-card .brand { font-size:11px; color:#64748b; margin-top:4px; }
      .demo-card .price { font-size:15px; color:#0f172a; font-weight:700; margin-top:6px; }
      .demo-card .badge { position:absolute; top:8px; right:8px; padding:3px 8px; border-radius:10px; font-size:11px; font-weight:700; }
      .badge.matched { background:#059669; color:#fff; }
      .badge.nomatch { background:#e2e8f0; color:#64748b; }
      .badge.fail    { background:#fef2f2; color:#dc2626; }
    </style>
    <div class='demo-grid'>
    """]
    for i, c in enumerate(cands):
        cls = ""
        badge = ""
        if verdicts:
            v = verdicts[i]
            if v.status != "OK":
                badge = f"<span class='badge fail'>ERR</span>"
            elif v.matched:
                cls = "matched"
                badge = "<span class='badge matched'>MATCH</span>"
            else:
                cls = "nomatch"
                badge = "<span class='badge nomatch'>—</span>"
        img = c.image_url or ""
        html.append(f"""
        <div class='demo-card {cls}'>
          {badge}
          {('<img src="' + img + '" />') if img else '<div style="height:140px;background:#f8fafc;border-radius:8px;"></div>'}
          <div class='brand'>{c.brand or '·'}</div>
          <div class='title'>{c.title}</div>
          <div class='price'>{c.price}</div>
        </div>
        """)
    html.append("</div>")
    display(HTML("".join(html)))

render_candidates(candidates)

## Step 4 — 학습된 모델에 매칭 판정 요청

각 후보에 대해 `(sku_name, candidate.title)` 쌍을 보내 matched/not_matched 응답.
`matched` 후보는 녹색으로 하이라이트된다.

In [ ]:
verdicts = []
for i, c in enumerate(candidates):
    v = model.predict(q["sku_name"], c.title)
    verdicts.append(v)
    status = "✓ MATCH" if v.matched else ("✗"
             if v.status == "OK" else f"!{v.status}")
    print(f"  [{i:02d}] {status:8s} ({v.latency_s*1000:.0f}ms)  {c.title[:60]}")

n_match = sum(1 for v in verdicts if v.matched)
print(f"\n▶ 총 {len(verdicts)} 후보 중 {n_match}건 matched")

render_candidates(candidates, verdicts)

## Step 5 — 정리

In [ ]:
try:
    browser.close()
    playwright.stop()
except Exception:
    pass
print("✓ closed")